# N023 · 字符串解析与有限状态机

**目标：** 按状态和合法转移解析含符号与边界的输入。

**先修：** N002, N006, N016。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 空白；符号；数字扫描；32位截断；状态转移；非法后缀。

## 从问题到算法

解析数字不是“看到数字就累加”。先区分前导空白、可选符号、数字前缀，再处理截断和范围；科学计数法还需要记录小数点、指数以及指数后是否真的出现数字。本章字符范围限定为 ASCII 数字。`my_atoi`允许停止于第一个非法字符；`is_number`要求整个去首尾空白后的串合法。

## 最小实现

**本章接口：** `my_atoi(s)`、`is_number(s)`

In [1]:
def my_atoi(s):
    i=0; n=len(s)
    while i<n and s[i]==' ': i+=1
    sign=1
    if i<n and s[i] in '+-':
        sign=-1 if s[i]=='-' else 1; i+=1
    value=0
    while i<n and '0'<=s[i]<='9':
        value=value*10+ord(s[i])-ord('0'); i+=1
    return max(-(1<<31),min((1<<31)-1,sign*value))

def is_number(s):
    s=s.strip()
    if not s: return False
    seen_digit=False; seen_dot=False; seen_exp=False
    for i,c in enumerate(s):
        if '0'<=c<='9': seen_digit=True
        elif c in '+-':
            if i!=0 and s[i-1] not in 'eE': return False
        elif c=='.':
            if seen_dot or seen_exp: return False
            seen_dot=True
        elif c in 'eE':
            if seen_exp or not seen_digit: return False
            seen_exp=True; seen_digit=False
        else: return False
    return seen_digit

## 正确性、前提与复杂度

指数前必须已有数字，读到指数后重置seen_digit，最终要求指数部分有数字；小数点只能在指数之前出现一次，符号只允许开头或紧跟指数。

**代价：** O(n) 字符扫描；atoi逐步大整数计算存在位代价，可在超界确定后提前饱和以限制位数。本实现为教学直写。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['输入','atoi前缀解析','完整数字判定'],[(s,my_atoi(s),is_number(s)) for s in ['  -42','4193 with words','3.5e2','1e','+','.8','9e-3']])

输入,atoi前缀解析,完整数字判定
-42,-42,True
4193 with words,4193,False
3.5e2,3,True
1e,1,False
+,0,False
.8,0,True
9e-3,9,True


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert my_atoi('')==0 and my_atoi('  -42')==-42
assert my_atoi('4193 with words')==4193
assert my_atoi('91283472332')==(1<<31)-1
assert my_atoi('-91283472332')==-(1<<31)
for s in ['1e','e3','+','.','1e+','1.2.3','6e-']: assert not is_number(s)
for s in ['2','-0.1','2e10','.8','3.','+6e-1']: assert is_number(s)
print('N023: 所有本章断言通过')

N023: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 区分完整数字校验和前缀转整数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 为指数符号设计状态。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 8. String to Integer (atoi)（canonical）
- 65. Valid Number（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。